# Módulo D: Interpretabilidad e integración (adaptado al Módulo B)

El enunciado plantea SHAP sobre el Módulo A; como el equipo no desarrolla A, aquí se interpreta el
**LightGBM de series de tiempo del Módulo B** (rezagos, medias móviles, calendario).

1. **SHAP:** importancia global (beeswarm), dependencia de las 3 variables más relevantes,
   explicación local de un día bien pronosticado y de un día con gran error; contraste con
   importancia por permutación.
2. **Integración:** del pronóstico diario a un plan de capacidad por estado de origen.
3. **Caso de uso y limitaciones.**

**Responsable:** _(nombre)_

In [ ]:
# Configuración común: imports, conexión solo lectura y estilo de gráficos
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from olist import conectar, consulta, RANDOM_STATE
from olist import config
from olist.graficos import estilo, guardar

estilo()
np.random.seed(RANDOM_STATE)
con = conectar()  # read_only=True: la base cruda no se modifica

from olist import series as ts
import joblib
import shap
from sklearn.inspection import permutation_importance

## 0. Datos y modelo del Módulo B

In [ ]:
df = consulta(con, "B_series/serie_diaria.sql", [config.FECHA_INICIO_UTIL, config.FECHA_FIN_UTIL])
y = df.set_index(pd.DatetimeIndex(df["fecha"]).rename("fecha"))["ordenes"].astype(float).asfreq("D")
y_trval, y_prueba = ts.separar_prueba(y)

ruta_modelo = config.MODELS_DIR / "lightgbm_series.joblib"
if ruta_modelo.exists():
    modelo = joblib.load(ruta_modelo)  # generado por B_series.ipynb
else:
    modelo = ts.entrenar_lightgbm(y_trval, random_state=RANDOM_STATE)

X_trval, _ = ts.matriz_supervisada(y_trval)
# Para explicar la prueba se usan los rezagos REALES (pronóstico a 1 paso) sin reentrenar
X_todo, y_todo = ts.matriz_supervisada(y)
X_prueba, y_obj_prueba = X_todo.loc[y_prueba.index], y_todo.loc[y_prueba.index]
pred_prueba = pd.Series(modelo.predict(X_prueba), index=X_prueba.index)
print("Métricas a 1 paso en prueba:", ts.metricas(y_obj_prueba, pred_prueba))

## 1. SHAP: importancia global

In [ ]:
explicador = shap.TreeExplainer(modelo)
shap_prueba = explicador(X_prueba)
shap.plots.beeswarm(shap_prueba, max_display=15, show=False)
guardar(plt.gcf(), "shap_beeswarm", "D_integracion")
plt.show()

## 2. Dependencia de las 3 variables más relevantes

In [ ]:
importancia = pd.Series(np.abs(shap_prueba.values).mean(axis=0), index=X_prueba.columns).sort_values(ascending=False)
top3 = importancia.index[:3].tolist()
print("Top 3:", top3)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, var in zip(axes, top3):
    shap.plots.scatter(shap_prueba[:, var], ax=ax, show=False)
guardar(fig, "shap_dependencia_top3", "D_integracion")
plt.show()

## 3. Explicaciones locales

Análogo al par "orden detectada / falso negativo" del enunciado: el **día mejor pronosticado** y el
**día con mayor error** del período de prueba.

In [ ]:
error = (y_obj_prueba - pred_prueba).abs()
dia_bueno, dia_malo = error.idxmin(), error.idxmax()
for etiqueta, dia in [("mejor pronosticado", dia_bueno), ("mayor error", dia_malo)]:
    i = X_prueba.index.get_loc(dia)
    print(f"{etiqueta}: {dia.date()}  real={y_obj_prueba[dia]:.0f}  pred={pred_prueba[dia]:.0f}")
    shap.plots.waterfall(shap_prueba[i], max_display=12, show=False)
    guardar(plt.gcf(), f"shap_local_{etiqueta.replace(' ', '_')}", "D_integracion")
    plt.show()

**TODO:** leer las explicaciones: ¿qué variables empujan el pronóstico? ¿por qué falla el peor día (festivo, paro, cambio de nivel)?

## 4. Contraste con importancia por permutación

In [ ]:
perm = permutation_importance(modelo, X_prueba, y_obj_prueba, n_repeats=20,
                              random_state=RANDOM_STATE, scoring="neg_mean_absolute_error")
comparacion = pd.DataFrame({
    "shap_medio_abs": importancia,
    "permutacion_MAE": pd.Series(perm.importances_mean, index=X_prueba.columns),
}).sort_values("shap_medio_abs", ascending=False)
comparacion.head(15)

**TODO:** discutir coincidencias y diferencias (variables correlacionadas como `lag_7` y `media_7` se reparten la importancia).

## 5. Integración: plan de capacidad semanal por estado de origen

Se combina el pronóstico del Módulo B con la participación histórica de cada estado de origen de los
vendedores (SQL), para estimar la carga de despachos por estado en las próximas 4 semanas.

In [ ]:
pronostico = ts.pronostico_recursivo(modelo, y_trval, config.HORIZONTE)
# ventana histórica: las 12 semanas anteriores al origen del pronóstico
fin = (y_trval.index[-1] + pd.Timedelta(days=1)).date().isoformat()
inicio = (y_trval.index[-1] - pd.Timedelta(weeks=12) + pd.Timedelta(days=1)).date().isoformat()
participacion = consulta(con, "D_integracion/participacion_estado_vendedor.sql", [inicio, fin])

# bloques de 7 días completos desde el primer día pronosticado (4 semanas)
semanal = pronostico.resample("7D").sum()
plan = pd.DataFrame(
    np.outer(semanal.to_numpy(), participacion["participacion"].to_numpy()),
    index=pd.Index(semanal.index.date, name="semana_desde"), columns=participacion["estado_vendedor"],
).round(0)
plan.iloc[:, :8]  # estados con más despachos

**TODO (opcional, otras integraciones):**
- ¿El modelo global sirve para São Paulo? → `sql/D_integracion/serie_diaria_estado.sql` con `'SP'`.
- ¿Mejora el pronóstico al agregar variables rezagadas de los datos (precio medio, flete, mezcla de categorías)?

## 6. Caso de uso y limitaciones

**TODO:**
- Operación: pronóstico semanal (cada lunes) → plan de capacidad por estado; alerta si el pronóstico supera la capacidad.
- Limitaciones: deriva temporal, censura al cierre del dataset (últimos meses incompletos),
  sesgos de cobertura geográfica (concentración en SP), eventos no recurrentes (paro de camioneros 2018).

In [ ]:
con.close()